# Lesson 20 — Capstone (Intermediate)

## Goal

Implement a **piece of an optimization solver**, plug it into `discopt`, and
benchmark against the default. Three options:

### A) A new branching rule (research-style)

Implement strong branching (Lesson 14) with a custom score function — e.g.,
*hybrid* (max $f_j (1-f_j)$ × pseudocost). `discopt`'s stable solve API
exposes `branching_policy="fractional" | "gnn"`; a public callback hook is
not yet stable, so a real plug-in either (a) requires a fork of the Rust
backend, or (b) you can fall back to *post-hoc analysis*: drive the solver
to specific node-orderings via warm starts and `time_limit`, and infer
what a custom rule would do. Either route counts; document which you took.
Benchmark against the default on ≥ 5 medium MILPs from
`discopt_benchmarks/instances/`.

Deliverables: rule code, results table (nodes, time, gap), and discussion.

### B) A new cut family (research-style)

Implement *zero-half cuts* (a special case of MIR, generated via parity
arguments). A stable cut-callback API is also not exposed; route (a) above
applies. Alternatively, generate cuts *offline* and add them as constraints
on a per-instance basis, then re-solve — the speedup vs the cut-free
formulation is the headline number. Benchmark.

Deliverables: cut code, separation routine, results table.

### C) A new decomposition

Pick one structured problem (network design, multi-commodity flow,
stochastic LP) and implement Benders or column generation from scratch.
Compare runtime and gap to monolithic solve.

## What to deliver

- `writing_response.ipynb` with code, plots, and result tables.
- `writing_response.md` (~2,000 words) describing your design,
  experiments, and conclusions.

## Scope and benchmarking protocol

Aim for **15–20 hours**. Document each design decision; cite at least 5
references; include 95 % bootstrap CIs on timing comparisons (see
`discopt_benchmarks/utils/statistics.py`).

**Benchmarking protocol** — apply this to every numerical comparison:

- **Instances:** at least 5 instances drawn from
  `discopt_benchmarks/instances/<problem-class>/`, where `<problem-class>` is
  whichever subfolder matches your project (e.g., `knapsack/`,
  `facility-location/`, `stochastic/`). All instances must come from the same
  subfolder for like-for-like comparison.
- **Repetitions:** 5 runs per (instance, configuration) pair with different
  random seeds (use `--seed` if your callback uses randomness).
- **Effect-size threshold:** "method A beats method B" requires a
  Wilcoxon signed-rank test ($p < 0.05$) on per-instance log-runtime *and*
  a median speedup of $\ge 1.2\times$. Anything weaker than that is
  reported as "no significant difference."
- **Failure handling:** any instance that hits the time limit counts as the
  time limit *plus* the optimality gap reported at termination; don't drop
  unsolved instances from the average.

## References
Pick from {cite:p}`Achterberg2005,Achterberg2007,Cornuejols2008,Benders1962,DantzigWolfe1960,Wolsey1998` based on your project.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm
